# Understanding Hyperparameters in Deep Learning

This notebook provides a comprehensive guide to understanding hyperparameters in deep learning, including what they are, their importance, and techniques for optimizing them.

## Table of Contents
1. [Setup and Required Libraries](#setup)
2. [What are Hyperparameters?](#what-are-hyperparameters)
3. [Common Neural Network Hyperparameters](#common-hyperparameters)
4. [Learning Rate Exploration](#learning-rate)
5. [Batch Size Effects](#batch-size)
6. [Optimization Algorithm Comparison](#optimizers)
7. [Regularization Techniques](#regularization)
8. [Network Architecture Hyperparameters](#architecture)
9. [Hyperparameter Tuning Methods](#tuning-methods)
10. [Case Study: Hyperparameter Optimization](#case-study)

## 1. Setup and Required Libraries <a name="setup"></a>

Let's start by installing and importing all the necessary packages for our hyperparameter exploration:

In [ ]:
# Install required packages
!pip install tensorflow scikit-learn matplotlib pandas numpy seaborn keras-tuner optuna

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

# Deep learning and machine learning libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, TensorBoard
from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Hyperparameter tuning libraries
import keras_tuner as kt
import optuna

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check if GPU is available
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. What are Hyperparameters? <a name="what-are-hyperparameters"></a>

Hyperparameters are configuration settings used to control the learning process of a machine learning or deep learning algorithm. Unlike model parameters (such as weights and biases) that are learned during training, hyperparameters must be set before training begins.

### Key Differences Between Parameters and Hyperparameters:

| Parameters | Hyperparameters |
|------------|----------------|
| Learned during training | Set before training |
| Updated by optimization algorithms | Manually tuned or automatically optimized |
| Internal values (weights, biases) | External configuration values |
| Saved as part of the model | Control the training process |

Let's demonstrate the impact of hyperparameters by building a simple model with different hyperparameter configurations:

In [ ]:
# Load a dataset for demonstration
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocess the data
X_train = X_train.reshape(-1, 28*28).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28*28).astype('float32') / 255.0

# Convert labels to categorical
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

# Split training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape: {X_test.shape}")

In [ ]:
def create_model(learning_rate=0.001, hidden_units=128, activation='relu', dropout_rate=0.2):
    """Create a simple neural network with specified hyperparameters"""
    model = keras.Sequential([
        layers.Dense(hidden_units, activation=activation, input_shape=(784,)),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units//2, activation=activation),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Example of how hyperparameters affect model structure and output
model_default = create_model()
model_default.summary()

In [ ]:
# Track models with different hyperparameters
models = {}
histories = {}

# Configuration 1: Default
models['default'] = create_model()
histories['default'] = models['default'].fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_val, y_val),
    verbose=1
)

# Configuration 2: Higher learning rate
models['high_lr'] = create_model(learning_rate=0.01)
histories['high_lr'] = models['high_lr'].fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_val, y_val),
    verbose=1
)

# Configuration 3: More hidden units
models['large'] = create_model(hidden_units=512)
histories['large'] = models['large'].fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_val, y_val),
    verbose=1
)

# Configuration 4: Different activation
models['tanh'] = create_model(activation='tanh')
histories['tanh'] = models['tanh'].fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
# Compare validation accuracy between different models
plt.figure(figsize=(12, 6))

for name, history in histories.items():
    plt.plot(history.history['val_accuracy'], label=f'{name}')

plt.title('Validation Accuracy with Different Hyperparameters')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Evaluate final accuracy
results = {}
for name, model in models.items():
    results[name] = model.evaluate(X_test, y_test, verbose=0)[1]
    
# Plot final results
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values())
plt.title('Test Accuracy with Different Hyperparameters')
plt.ylabel('Accuracy')
plt.ylim(0.9, 1.0)  # Zoom in to see differences
plt.grid(axis='y')
for i, (k, v) in enumerate(results.items()):
    plt.text(i, v-0.01, f'{v:.4f}', ha='center')
plt.show()

## 3. Common Neural Network Hyperparameters <a name="common-hyperparameters"></a>

Let's explore the most important hyperparameters in deep learning:

### 3.1 Learning-related Hyperparameters
- **Learning Rate**: Controls the step size during optimization
- **Batch Size**: Number of samples processed before model update
- **Epochs**: Number of complete passes through the dataset
- **Optimizer**: Algorithm used to update the weights (SGD, Adam, RMSprop, etc.)

### 3.2 Network Architecture Hyperparameters
- **Number of Layers**: Depth of the network
- **Units per Layer**: Width of each layer
- **Activation Functions**: Functions applied to layer outputs (ReLU, sigmoid, tanh, etc.)

### 3.3 Regularization Hyperparameters
- **Dropout Rate**: Fraction of units to drop during training
- **L1/L2 Regularization**: Penalty terms added to the loss function
- **Early Stopping Criteria**: When to stop training to prevent overfitting

Let's create a visual representation of these hyperparameters:

In [ ]:
# Create a table of common hyperparameters with their ranges and effects
hyperparameters = {
    'Hyperparameter': [
        'Learning Rate', 
        'Batch Size', 
        'Epochs', 
        'Number of Layers', 
        'Units per Layer',
        'Activation Function',
        'Optimizer',
        'Dropout Rate',
        'L1/L2 Regularization',
        'Weight Initialization'
    ],
    'Typical Range': [
        '0.0001 - 0.1',
        '16 - 512',
        '10 - 1000+',
        '1 - 100+',
        '32 - 1024',
        'ReLU, Tanh, Sigmoid, etc.',
        'SGD, Adam, RMSprop, etc.',
        '0.1 - 0.5',
        '0.0001 - 0.01',
        'Glorot/Xavier, He, etc.'
    ],
    'Effect on Model': [
        'Controls step size in gradient descent',
        'Affects training speed and generalization',
        'Determines total training iterations',
        'Controls model capacity and depth',
        'Controls model capacity and width',
        'Affects gradient flow and representation capacity',
        'Determines optimization strategy',
        'Controls overfitting',
        'Penalizes large weights to prevent overfitting',
        'Affects initial training dynamics'
    ]
}

df_hyperparams = pd.DataFrame(hyperparameters)
df_hyperparams

## 4. Learning Rate Exploration <a name="learning-rate"></a>

The learning rate is arguably the most important hyperparameter in deep learning. It controls how quickly or slowly the model's weights are updated during training.

### Effects of different learning rates:
- **Too high**: Model may diverge or oscillate around the minimum
- **Too low**: Training may be too slow or get stuck in local minima
- **Just right**: Model converges efficiently to a good solution

Let's explore how different learning rates affect training:

In [ ]:
# Test different learning rates
learning_rates = [0.0001, 0.001, 0.01, 0.1]
lr_histories = {}

for lr in learning_rates:
    model = create_model(learning_rate=lr)
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        validation_data=(X_val, y_val),
        verbose=0
    )
    lr_histories[f'lr={lr}'] = history

# Visualize training curves for different learning rates
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for name, history in lr_histories.items():
    plt.plot(history.history['loss'], label=name)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for name, history in lr_histories.items():
    plt.plot(history.history['val_loss'], label=name)
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy
plt.subplot(2, 2, 3)
for name, history in lr_histories.items():
    plt.plot(history.history['accuracy'], label=name)
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 4)
for name, history in lr_histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Learning Rate Schedules

Learning rate schedules change the learning rate during training. Common approaches include:
1. **Step Decay**: Reduce learning rate by a factor after specific epochs
2. **Exponential Decay**: Continuously decrease learning rate exponentially
3. **Cosine Annealing**: Cycle learning rate following a cosine function

Let's implement these schedules:

In [ ]:
# Define learning rate schedules
def step_decay(epoch, initial_lr=0.01):
    """Step decay: drop learning rate by half every 2 epochs"""
    drop_rate = 0.5
    epochs_drop = 2.0
    return initial_lr * np.power(drop_rate, np.floor((1+epoch)/epochs_drop))

def exp_decay(epoch, initial_lr=0.01):
    """Exponential decay: continuously decreasing learning rate"""
    k = 0.2  # Decay rate
    return initial_lr * np.exp(-k*epoch)

def cosine_decay(epoch, initial_lr=0.01, total_epochs=10):
    """Cosine annealing: cyclic learning rate following a cosine function"""
    return initial_lr * (1 + np.cos(np.pi * epoch / total_epochs)) / 2

# Plot learning rate schedules
epochs = range(10)
plt.figure(figsize=(12, 6))

plt.plot(epochs, [step_decay(e) for e in epochs], 'ro-', label='Step Decay')
plt.plot(epochs, [exp_decay(e) for e in epochs], 'bo-', label='Exponential Decay')
plt.plot(epochs, [cosine_decay(e) for e in epochs], 'go-', label='Cosine Annealing')

plt.title('Learning Rate Schedules')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Train models with different learning rate schedules
lr_schedule_histories = {}
schedules = {
    'step_decay': LearningRateScheduler(step_decay),
    'exp_decay': LearningRateScheduler(exp_decay),
    'cosine_decay': LearningRateScheduler(cosine_decay),
    'constant': None  # No scheduler, constant learning rate
}

for name, scheduler in schedules.items():
    model = create_model(learning_rate=0.01)
    callbacks = [scheduler] if scheduler else []
    
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        validation_data=(X_val, y_val),
        verbose=0,
        callbacks=callbacks
    )
    lr_schedule_histories[name] = history

# Plot results with different learning rate schedules
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for name, history in lr_schedule_histories.items():
    plt.plot(history.history['loss'], label=name)
plt.title('Training Loss with Different LR Schedules')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for name, history in lr_schedule_histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy with Different LR Schedules')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Batch Size Effects <a name="batch-size"></a>

Batch size determines how many samples are processed before the model weights are updated. It has significant impacts on:
- Training speed
- Memory usage
- Generalization performance
- Convergence behavior

### Trade-offs with batch size:

- **Small batch size** (e.g., 8-32):
  - More frequent updates
  - More noise in gradient (can help escape local minima)
  - Less memory intensive
  - Slower overall training (more iterations per epoch)
  - Often better generalization

- **Large batch size** (e.g., 256-1024):
  - More accurate gradient estimates
  - Faster training in terms of epochs
  - Better utilizes GPU parallelism
  - More memory intensive
  - May lead to poorer generalization

Let's compare different batch sizes:

In [ ]:
# Test different batch sizes
batch_sizes = [16, 64, 256, 1024]
batch_histories = {}

for bs in batch_sizes:
    model = create_model()
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=bs,
        validation_data=(X_val, y_val),
        verbose=0
    )
    training_time = time.time() - start_time
    
    batch_histories[f'batch_size={bs}'] = {
        'history': history,
        'time': training_time
    }

# Plot training/validation metrics for different batch sizes
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for name, data in batch_histories.items():
    plt.plot(data['history'].history['loss'], label=name)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for name, data in batch_histories.items():
    plt.plot(data['history'].history['val_loss'], label=name)
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 3)
for name, data in batch_histories.items():
    plt.plot(data['history'].history['val_accuracy'], label=name)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot training time
plt.subplot(2, 2, 4)
names = [f'bs={bs}' for bs in batch_sizes]
times = [data['time'] for data in batch_histories.values()]
plt.bar(names, times)
plt.title('Training Time (5 epochs)')
plt.ylabel('Time (seconds)')
plt.grid(True, axis='y')
for i, t in enumerate(times):
    plt.text(i, t+0.1, f'{t:.2f}s', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# More detailed analysis of batch size impact
# We'll track iterations vs. epochs

# For smaller dataset to make it faster
subset_indices = np.random.choice(len(X_train), 10000, replace=False)
X_train_subset = X_train[subset_indices]
y_train_subset = y_train[subset_indices]

# Calculate iterations per epoch for each batch size
iterations_per_epoch = {bs: int(np.ceil(len(X_train_subset) / bs)) for bs in [16, 64, 256]}

# Custom callback to track metrics after every batch
class MetricTracker(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.batch_losses = []
        self.batch_accuracies = []
        self.iteration_counts = []
        self.current_iteration = 0
        
    def on_batch_end(self, batch, logs=None):
        self.batch_losses.append(logs['loss'])
        self.batch_accuracies.append(logs['accuracy'])
        self.iteration_counts.append(self.current_iteration)
        self.current_iteration += 1

# Train models with different batch sizes and track batch-level metrics
detailed_batch_histories = {}

for bs in [16, 64, 256]:
    model = create_model()
    tracker = MetricTracker()
    
    history = model.fit(
        X_train_subset, y_train_subset,
        epochs=3,
        batch_size=bs,
        validation_data=(X_val, y_val),
        verbose=0,
        callbacks=[tracker]
    )
    
    detailed_batch_histories[bs] = {
        'batch_losses': tracker.batch_losses,
        'batch_accuracies': tracker.batch_accuracies,
        'iteration_counts': tracker.iteration_counts,
        'iterations_per_epoch': iterations_per_epoch[bs]
    }

# Plot batch-level metrics
plt.figure(figsize=(15, 6))

# Plot training loss per iteration
plt.subplot(1, 2, 1)
for bs, data in detailed_batch_histories.items():
    iterations = data['iteration_counts']
    losses = data['batch_losses']
    # Add vertical lines to show epoch boundaries
    for i in range(1, 3):
        plt.axvline(x=i*data['iterations_per_epoch'], color='gray', linestyle='--', alpha=0.5)
    plt.plot(iterations, losses, label=f'Batch Size={bs}')

plt.title('Training Loss per Iteration')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy per iteration
plt.subplot(1, 2, 2)
for bs, data in detailed_batch_histories.items():
    iterations = data['iteration_counts']
    accuracies = data['batch_accuracies']
    # Add vertical lines to show epoch boundaries
    for i in range(1, 3):
        plt.axvline(x=i*data['iterations_per_epoch'], color='gray', linestyle='--', alpha=0.5)
    plt.plot(iterations, accuracies, label=f'Batch Size={bs}')

plt.title('Training Accuracy per Iteration')
plt.xlabel('Iteration')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 6. Optimization Algorithm Comparison <a name="optimizers"></a>

The choice of optimization algorithm significantly affects the training dynamics and final model performance. Common optimizers include:

- **Stochastic Gradient Descent (SGD)**: Classic algorithm, often with momentum
- **Adam**: Adaptive learning rates, combines momentum and RMSprop
- **RMSprop**: Adapts learning rates based on recent gradients
- **Adagrad**: Adapts learning rate per parameter
- **Adamax**: Adam variant with infinity norm

Let's compare their performance:

In [ ]:
# Test different optimizers
optimizers = {
    'SGD': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Adam': keras.optimizers.Adam(learning_rate=0.001),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.001),
    'Adagrad': keras.optimizers.Adagrad(learning_rate=0.01),
    'Adamax': keras.optimizers.Adamax(learning_rate=0.001)
}

optimizer_histories = {}

for name, opt in optimizers.items():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        validation_data=(X_val, y_val),
        verbose=0
    )
    
    optimizer_histories[name] = history

# Create visualization of optimizer performance
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for name, history in optimizer_histories.items():
    plt.plot(history.history['loss'], label=name)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for name, history in optimizer_histories.items():
    plt.plot(history.history['val_loss'], label=name)
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy
plt.subplot(2, 2, 3)
for name, history in optimizer_histories.items():
    plt.plot(history.history['accuracy'], label=name)
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 4)
for name, history in optimizer_histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize optimizer behavior in a 2D loss landscape

# Create a simple 2D function to optimize
def loss_function(x, y):
    # Himmelblau's function - has multiple local minima
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

# Create grid for visualization
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = loss_function(X, Y)

# Initial points for different optimizers
init_x, init_y = 4.0, 4.0

# Optimization paths for different optimizers
steps = 50
learning_rate = 0.01

# Initialize paths
paths = {
    'SGD': {'x': [init_x], 'y': [init_y]},
    'SGD with Momentum': {'x': [init_x], 'y': [init_y]},
    'Adam': {'x': [init_x], 'y': [init_y]},
    'RMSprop': {'x': [init_x], 'y': [init_y]}
}

# SGD
x_sgd, y_sgd = init_x, init_y
for _ in range(steps):
    # Compute gradients
    dx = 2 * (x_sgd**2 + y_sgd - 11) * 2 * x_sgd + 2 * (x_sgd + y_sgd**2 - 7)
    dy = 2 * (x_sgd**2 + y_sgd - 11) + 2 * (x_sgd + y_sgd**2 - 7) * 2 * y_sgd
    
    # Update parameters
    x_sgd -= learning_rate * dx
    y_sgd -= learning_rate * dy
    
    # Store
    paths['SGD']['x'].append(x_sgd)
    paths['SGD']['y'].append(y_sgd)

# SGD with Momentum
x_mom, y_mom = init_x, init_y
vx, vy = 0, 0  # Velocities
beta = 0.9  # Momentum parameter
for _ in range(steps):
    # Compute gradients
    dx = 2 * (x_mom**2 + y_mom - 11) * 2 * x_mom + 2 * (x_mom + y_mom**2 - 7)
    dy = 2 * (x_mom**2 + y_mom - 11) + 2 * (x_mom + y_mom**2 - 7) * 2 * y_mom
    
    # Update velocities with momentum
    vx = beta * vx - learning_rate * dx
    vy = beta * vy - learning_rate * dy
    
    # Update parameters
    x_mom += vx
    y_mom += vy
    
    # Store
    paths['SGD with Momentum']['x'].append(x_mom)
    paths['SGD with Momentum']['y'].append(y_mom)

# Adam
x_adam, y_adam = init_x, init_y
m_x, m_y = 0, 0  # First moment estimates
v_x, v_y = 0, 0  # Second moment estimates
beta1, beta2 = 0.9, 0.999  # Adam parameters
eps = 1e-8
for t in range(1, steps+1):
    # Compute gradients
    dx = 2 * (x_adam**2 + y_adam - 11) * 2 * x_adam + 2 * (x_adam + y_adam**2 - 7)
    dy = 2 * (x_adam**2 + y_adam - 11) + 2 * (x_adam + y_adam**2 - 7) * 2 * y_adam
    
    # Update biased first moment estimates
    m_x = beta1 * m_x + (1 - beta1) * dx
    m_y = beta1 * m_y + (1 - beta1) * dy
    
    # Update biased second moment estimates
    v_x = beta2 * v_x + (1 - beta2) * dx**2
    v_y = beta2 * v_y + (1 - beta2) * dy**2
    
    # Bias correction
    m_x_hat = m_x / (1 - beta1**t)
    m_y_hat = m_y / (1 - beta1**t)
    v_x_hat = v_x / (1 - beta2**t)
    v_y_hat = v_y / (1 - beta2**t)
    
    # Update parameters
    x_adam -= learning_rate * m_x_hat / (np.sqrt(v_x_hat) + eps)
    y_adam -= learning_rate * m_y_hat / (np.sqrt(v_y_hat) + eps)
    
    # Store
    paths['Adam']['x'].append(x_adam)
    paths['Adam']['y'].append(y_adam)

# RMSprop
x_rms, y_rms = init_x, init_y
g_x, g_y = 0, 0  # Accumulated squared gradients
rho = 0.9  # RMSprop decay factor
for _ in range(steps):
    # Compute gradients
    dx = 2 * (x_rms**2 + y_rms - 11) * 2 * x_rms + 2 * (x_rms + y_rms**2 - 7)
    dy = 2 * (x_rms**2 + y_rms - 11) + 2 * (x_rms + y_rms**2 - 7) * 2 * y_rms
    
    # Update accumulated squared gradients
    g_x = rho * g_x + (1 - rho) * dx**2
    g_y = rho * g_y + (1 - rho) * dy**2
    
    # Update parameters
    x_rms -= learning_rate * dx / (np.sqrt(g_x) + eps)
    y_rms -= learning_rate * dy / (np.sqrt(g_y) + eps)
    
    # Store
    paths['RMSprop']['x'].append(x_rms)
    paths['RMSprop']['y'].append(y_rms)

# Plot 2D loss landscape with optimization paths
plt.figure(figsize=(14, 12))

# Contour plot of the loss landscape
contour_levels = np.logspace(0, 3, 20)
plt.contour(X, Y, Z, levels=contour_levels, cmap='viridis', alpha=0.7)
plt.colorbar(label='Loss Value')

# Plot optimization paths
for name, path in paths.items():
    plt.plot(path['x'], path['y'], 'o-', label=name, linewidth=2, markersize=3)

# Mark initial point and global minima
plt.scatter([init_x], [init_y], color='red', s=100, marker='*', label='Initial Point')

# Known minima of Himmelblau's function
minima = [
    (3.0, 2.0),
    (-2.81, 3.13),
    (-3.78, -3.28),
    (3.58, -1.85)
]
for i, (x_min, y_min) in enumerate(minima):
    plt.scatter([x_min], [y_min], color='green', s=100, marker='X')
    if i == 0:
        plt.annotate('Local Minima', (x_min, y_min), xytext=(x_min + 0.5, y_min + 0.5),
                    arrowprops=dict(facecolor='black', shrink=0.05))

plt.title('Optimization Paths in 2D Loss Landscape')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

## 7. Regularization Techniques <a name="regularization"></a>

Regularization techniques help prevent overfitting by constraining the model's complexity. Common regularization hyperparameters include:

1. **Dropout rate**: Fraction of neurons randomly dropped during training
2. **L1/L2 regularization**: Penalties added to the loss function based on weights
3. **Early stopping**: Stopping training when validation metrics stop improving
4. **Batch normalization**: Normalizes layer inputs, reducing internal covariate shift

Let's explore how these regularization techniques affect model performance:

In [ ]:
# Create a more complex dataset prone to overfitting
# We'll use a subset of the data to make overfitting more likely
subset_size = 1000
X_small = X_train[:subset_size]
y_small = y_train[:subset_size]

# Create a more complex model architecture
def create_complex_model(dropout_rate=0.0, l2_lambda=0.0):
    """Create a more complex model with optional regularization"""
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(784,), 
                     kernel_regularizer=regularizers.l2(l2_lambda) if l2_lambda > 0 else None),
        layers.Dropout(dropout_rate),
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_lambda) if l2_lambda > 0 else None),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_lambda) if l2_lambda > 0 else None),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Train models with different regularization techniques
reg_histories = {}

# No regularization
model_no_reg = create_complex_model()
history_no_reg = model_no_reg.fit(
    X_small, y_small,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)
reg_histories['No Regularization'] = history_no_reg

# Dropout
model_dropout = create_complex_model(dropout_rate=0.5)
history_dropout = model_dropout.fit(
    X_small, y_small,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)
reg_histories['Dropout (0.5)'] = history_dropout

# L2 regularization
model_l2 = create_complex_model(l2_lambda=0.001)
history_l2 = model_l2.fit(
    X_small, y_small,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)
reg_histories['L2 Regularization'] = history_l2

# Dropout + L2 regularization
model_both = create_complex_model(dropout_rate=0.3, l2_lambda=0.0005)
history_both = model_both.fit(
    X_small, y_small,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)
reg_histories['Dropout + L2'] = history_both

# Early stopping
model_early = create_complex_model()
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
history_early = model_early.fit(
    X_small, y_small,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=0
)
reg_histories['Early Stopping'] = history_early

# Plot results
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for name, history in reg_histories.items():
    plt.plot(history.history['loss'], label=name)
plt.title('Training Loss with Different Regularization')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for name, history in reg_histories.items():
    plt.plot(history.history['val_loss'], label=name)
plt.title('Validation Loss with Different Regularization')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy
plt.subplot(2, 2, 3)
for name, history in reg_histories.items():
    plt.plot(history.history['accuracy'], label=name)
plt.title('Training Accuracy with Different Regularization')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 4)
for name, history in reg_histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy with Different Regularization')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Calculate final test accuracy for each model
models = {
    'No Regularization': model_no_reg,
    'Dropout (0.5)': model_dropout,
    'L2 Regularization': model_l2,
    'Dropout + L2': model_both,
    'Early Stopping': model_early
}

test_accuracies = {}
for name, model in models.items():
    _, accuracy = model.evaluate(X_test, y_test, verbose=0)
    test_accuracies[name] = accuracy

# Plot test accuracies
plt.figure(figsize=(10, 6))
plt.bar(test_accuracies.keys(), test_accuracies.values())
plt.title('Test Accuracy with Different Regularization Techniques')
plt.ylabel('Accuracy')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()

for i, (k, v) in enumerate(test_accuracies.items()):
    plt.text(i, v-0.02, f'{v:.4f}', ha='center', fontsize=10)
    
plt.show()

In [ ]:
# Visualization of how regularization affects model weights
plt.figure(figsize=(15, 10))

# For each model, extract weights from the first layer
model_names = ['No Regularization', 'Dropout (0.5)', 'L2 Regularization', 'Dropout + L2']
for i, name in enumerate(model_names):
    # Get first layer weights
    weights = models[name].layers[0].get_weights()[0]
    
    # Flatten the weights
    flat_weights = weights.flatten()
    
    # Plot histogram
    plt.subplot(2, 2, i+1)
    plt.hist(flat_weights, bins=50, alpha=0.7)
    plt.title(f'Weight Distribution - {name}')
    plt.xlabel('Weight Value')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    
    # Add statistics
    mean = np.mean(flat_weights)
    std = np.std(flat_weights)
    plt.axvline(mean, color='r', linestyle='--', alpha=0.8, label=f'Mean: {mean:.4f}')
    plt.text(0.02, 0.95, f'Mean: {mean:.4f}\nStd: {std:.4f}', 
             transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', alpha=0.1))

plt.tight_layout()
plt.show()

## 8. Network Architecture Hyperparameters <a name="architecture"></a>

The architecture of a neural network is defined by hyperparameters such as:
- Number of layers
- Units/neurons per layer
- Activation functions
- Skip connections (in advanced architectures)

These hyperparameters determine the network's capacity and ability to learn complex patterns.

Let's explore the impact of architectural choices:

In [ ]:
# Test different neural network architectures
def create_architecture(layers_config, activation='relu'):
    """Create a model with specified architecture"""
    model = keras.Sequential()
    
    # First layer needs input shape
    model.add(layers.Dense(layers_config[0], activation=activation, input_shape=(784,)))
    
    # Add remaining layers
    for units in layers_config[1:]:
        model.add(layers.Dense(units, activation=activation))
        
    # Output layer
    model.add(layers.Dense(10, activation='softmax'))
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Define architectures to test
architectures = {
    'Shallow (1 hidden)': [64],
    'Medium (2 hidden)': [128, 64],
    'Deep (4 hidden)': [256, 128, 64, 32],
    'Very Deep (6 hidden)': [256, 128, 64, 32, 16, 8]
}

# Train models with different architectures
arch_histories = {}

for name, arch in architectures.items():
    model = create_architecture(arch)
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        validation_data=(X_val, y_val),
        verbose=0
    )
    arch_histories[name] = history
    
    # Report parameters
    print(f"{name}: {model.count_params():,} parameters")

# Plot architecture comparison
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for name, history in arch_histories.items():
    plt.plot(history.history['loss'], label=name)
plt.title('Training Loss for Different Architectures')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for name, history in arch_histories.items():
    plt.plot(history.history['val_loss'], label=name)
plt.title('Validation Loss for Different Architectures')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy
plt.subplot(2, 2, 3)
for name, history in arch_histories.items():
    plt.plot(history.history['accuracy'], label=name)
plt.title('Training Accuracy for Different Architectures')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 4)
for name, history in arch_histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy for Different Architectures')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Test different activation functions
activations = ['relu', 'tanh', 'sigmoid', 'elu', 'selu']
act_histories = {}

for act in activations:
    model = create_architecture([128, 64], activation=act)
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        validation_data=(X_val, y_val),
        verbose=0
    )
    act_histories[act] = history

# Plot activation function comparison
plt.figure(figsize=(15, 10))

# Plot training loss
plt.subplot(2, 2, 1)
for act, history in act_histories.items():
    plt.plot(history.history['loss'], label=act)
plt.title('Training Loss for Different Activation Functions')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot validation loss
plt.subplot(2, 2, 2)
for act, history in act_histories.items():
    plt.plot(history.history['val_loss'], label=act)
plt.title('Validation Loss for Different Activation Functions')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training accuracy
plt.subplot(2, 2, 3)
for act, history in act_histories.items():
    plt.plot(history.history['accuracy'], label=act)
plt.title('Training Accuracy for Different Activation Functions')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot validation accuracy
plt.subplot(2, 2, 4)
for act, history in act_histories.items():
    plt.plot(history.history['val_accuracy'], label=act)
plt.title('Validation Accuracy for Different Activation Functions')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning Methods <a name="tuning-methods"></a>

Finding the optimal set of hyperparameters manually is challenging. Several methodologies exist for systematic hyperparameter tuning:

1. **Grid Search**: Exhaustively searching through all possible combinations of hyperparameter values
2. **Random Search**: Randomly sampling hyperparameter combinations, often more efficient than grid search
3. **Bayesian Optimization**: Building a probabilistic model of the objective function and using it to select promising hyperparameters
4. **Evolutionary Algorithms**: Using genetic algorithms to evolve hyperparameter combinations

Let's implement some of these approaches using popular libraries:

In [ ]:
# Let's implement grid search with scikit-learn
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

# Create a function that returns a compiled model
def create_model_for_grid(learning_rate=0.001, hidden_units=128, dropout_rate=0.2, activation='relu'):
    model = keras.Sequential([
        layers.Dense(hidden_units, activation=activation, input_shape=(784,)),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units//2, activation=activation),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# For demo purposes, we'll use a small subset of the data
X_sample = X_train[:5000].reshape(-1, 784)
y_sample = np.argmax(y_train[:5000], axis=1)  # Convert back to integer labels for sklearn

# Wrap the Keras model for scikit-learn
model = KerasClassifier(
    build_fn=create_model_for_grid,
    epochs=5,
    batch_size=128,
    verbose=0
)

# Define the grid search parameters
param_grid = {
    'learning_rate': [0.001, 0.01],
    'hidden_units': [64, 128],
    'dropout_rate': [0.2, 0.5],
    'activation': ['relu', 'tanh']
}

# Grid Search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,  # Use all available processors
    verbose=1
)

# This would take too long to run in this notebook
# grid_result = grid.fit(X_sample, y_sample)
print("Grid Search demo setup complete. In practice, this would run many models.")

In [ ]:
# Random search with Keras Tuner for a more realistic example
import keras_tuner as kt

def model_builder(hp):
    """Build model with hyperparameters from Keras Tuner"""
    model = keras.Sequential()
    
    # Tune number of layers and units
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=512, step=32),
            activation=hp.Choice(f'activation_{i}', ['relu', 'tanh']),
            kernel_regularizer=regularizers.l2(
                hp.Float(f'l2_{i}', 1e-5, 1e-2, sampling='log')
            )
        ))
        
        # Add dropout with tune-able rate
        model.add(layers.Dropout(
            hp.Float(f'dropout_{i}', 0, 0.5, step=0.1)
        ))
    
    # Output layer is fixed
    model.add(layers.Dense(10, activation='softmax'))
    
    # Tune learning rate
    learning_rate = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
    
    # Choose optimizer
    optimizer = hp.Choice('optimizer', ['adam', 'rmsprop', 'sgd'])
    if optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    
    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Initialize the tuner - Random Search
tuner = kt.RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=10,  # Number of different hyperparameter combinations to try
    executions_per_trial=1,  # Number of models to fit per trial (for variance reduction)
    directory='keras_tuner_dir',
    project_name='mnist_hyperparameter_tuning'
)

# Display search space summary
tuner.search_space_summary()

# We would run the search like this, but it takes too long for a notebook
# tuner.search(
#     X_train, y_train,
#     epochs=5,
#     batch_size=128,
#     validation_split=0.2
# )
print("\nIn practice, you would run tuner.search() to find optimal hyperparameters.")

In [ ]:
# Optuna example with visualization
import optuna

def objective(trial):
    """Optuna objective function to minimize"""
    # Hyperparameters to optimize
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    hidden_units = trial.suggest_categorical('hidden_units', [64, 128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop', 'sgd'])
    
    # Create and compile model
    model = keras.Sequential([
        layers.Dense(hidden_units, activation='relu', input_shape=(784,)),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units//2, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    
    # Select optimizer
    if optimizer_name == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
        
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train model
    # For real applications, use more epochs and a validation set
    # Here we keep it simple for illustration
    history = model.fit(
        X_train[:1000], y_train[:1000],
        epochs=3,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=0
    )
    
    # Return best validation accuracy (we want to maximize this)
    return max(history.history['val_accuracy'])

# In practice, you would run this:
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=100)

# For demonstration, let's create a mock study result
mock_study = optuna.create_study(direction='maximize')
# Add some mock trial results
mock_params = [
    {'learning_rate': 0.01, 'hidden_units': 128, 'dropout_rate': 0.3, 'batch_size': 64, 'optimizer': 'adam'},
    {'learning_rate': 0.005, 'hidden_units': 256, 'dropout_rate': 0.2, 'batch_size': 128, 'optimizer': 'adam'},
    {'learning_rate': 0.001, 'hidden_units': 64, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'rmsprop'},
    {'learning_rate': 0.02, 'hidden_units': 128, 'dropout_rate': 0.3, 'batch_size': 64, 'optimizer': 'sgd'},
    {'learning_rate': 0.003, 'hidden_units': 256, 'dropout_rate': 0.1, 'batch_size': 128, 'optimizer': 'adam'}
]
mock_values = [0.92, 0.94, 0.90, 0.89, 0.93]

# Add trials to our mock study
for params, value in zip(mock_params, mock_values):
    trial = optuna.trial.create_trial(
        params=params,
        distributions={
            'learning_rate': optuna.distributions.LogUniformDistribution(1e-4, 1e-1),
            'hidden_units': optuna.distributions.CategoricalDistribution([64, 128, 256]),
            'dropout_rate': optuna.distributions.UniformDistribution(0.1, 0.5),
            'batch_size': optuna.distributions.CategoricalDistribution([32, 64, 128]),
            'optimizer': optuna.distributions.CategoricalDistribution(['adam', 'rmsprop', 'sgd'])
        },
        value=value
    )
    mock_study.add_trial(trial)

# Display the results
print("Best trial:")
trial = mock_study.best_trial
print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Visualization of hyperparameter importance
try:
    # This requires matplotlib and plotly
    from optuna.visualization import plot_param_importances
    fig = plot_param_importances(mock_study)
    fig.show()
except Exception as e:
    print(f"Could not create visualization: {e}")
    print("Hyperparameter importance analysis requires additional dependencies.")
    
# Create a parallel coordinate plot (useful to see parameter interactions)
try:
    from optuna.visualization import plot_parallel_coordinate
    fig = plot_parallel_coordinate(mock_study)
    fig.show()
except Exception as e:
    print(f"Could not create parallel coordinate plot: {e}")

## 10. Case Study: Hyperparameter Optimization <a name="case-study"></a>

Let's put everything together in an end-to-end example. We'll optimize hyperparameters for a classification task using a systematic approach:

1. Define model architecture and hyperparameter space
2. Implement hyperparameter tuning with Keras Tuner
3. Track and visualize experiments
4. Select optimal configuration
5. Evaluate final model

In [ ]:
# Let's use Fashion MNIST for this case study to work with a different dataset
(fashion_x_train, fashion_y_train), (fashion_x_test, fashion_y_test) = fashion_mnist.load_data()

# Preprocess data
fashion_x_train = fashion_x_train.reshape(-1, 28*28).astype('float32') / 255.0
fashion_x_test = fashion_x_test.reshape(-1, 28*28).astype('float32') / 255.0

# Convert labels to categorical
fashion_y_train_cat = keras.utils.to_categorical(fashion_y_train, 10)
fashion_y_test_cat = keras.utils.to_categorical(fashion_y_test, 10)

# Split training data for validation
fashion_x_train, fashion_x_val, fashion_y_train_cat, fashion_y_val_cat = train_test_split(
    fashion_x_train, fashion_y_train_cat, test_size=0.2, random_state=42
)

print(f"Training data: {fashion_x_train.shape}")
print(f"Validation data: {fashion_x_val.shape}")
print(f"Test data: {fashion_x_test.shape}")

# Define class names for later visualization
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
# Define a model-building function for Keras Tuner
def fashion_model_builder(hp):
    """Model builder for Fashion MNIST with tunable hyperparameters"""
    model = keras.Sequential()
    
    # Tune the number of units in the first dense layer
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(784,)))
    
    # Tune dropout rate
    hp_dropout = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
    model.add(layers.Dropout(rate=hp_dropout))
    
    # Tune whether to add another dense layer and its units
    if hp.Boolean("add_layer"):
        hp_units_2 = hp.Int('units_2', min_value=16, max_value=256, step=16)
        model.add(layers.Dense(units=hp_units_2, activation='relu'))
        model.add(layers.Dropout(rate=hp_dropout))
    
    # Output layer (fixed)
    model.add(layers.Dense(10, activation='softmax'))
    
    # Tune learning rate
    hp_learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    
    # Compile the model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create the tuner
tuner = kt.BayesianOptimization(
    fashion_model_builder,
    objective='val_accuracy',
    max_trials=10,
    directory='fashion_mnist_tuning',
    project_name='hyperparameter_optimization_case_study'
)

# Display search space summary
tuner.search_space_summary()

# For demonstration purposes, we'll mock the tuning process results
print("\nIn a full implementation, we would run:")
print("tuner.search(fashion_x_train, fashion_y_train_cat, epochs=10, validation_data=(fashion_x_val, fashion_y_val_cat))")

# Mock best hyperparameters
best_hps = {
    'units': 128,
    'dropout': 0.2,
    'add_layer': True,
    'units_2': 64,
    'learning_rate': 0.001
}

print("\nBest hyperparameters found:")
for param, value in best_hps.items():
    print(f"- {param}: {value}")

# Build final model with the best hyperparameters
final_model = keras.Sequential([
    layers.Dense(best_hps['units'], activation='relu', input_shape=(784,)),
    layers.Dropout(best_hps['dropout']),
    layers.Dense(best_hps['units_2'], activation='relu'),
    layers.Dropout(best_hps['dropout']),
    layers.Dense(10, activation='softmax')
])

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_hps['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train final model
final_history = final_model.fit(
    fashion_x_train, fashion_y_train_cat,
    epochs=10,
    batch_size=64,
    validation_data=(fashion_x_val, fashion_y_val_cat)
)

# Evaluate final model
test_loss, test_acc = final_model.evaluate(fashion_x_test, fashion_y_test_cat)
print(f"\nFinal model test accuracy: {test_acc:.4f}")

# Plot training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(final_history.history['loss'], label='Training Loss')
plt.plot(final_history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(final_history.history['accuracy'], label='Training Accuracy')
plt.plot(final_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualize predictions on test set
def plot_image(i, predictions_array, true_labels, images):
    true_label = true_labels[i]
    img = images[i].reshape(28, 28)
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(img, cmap=plt.cm.binary)
    
    predicted_label = np.argmax(predictions_array[i])
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'
    
    plt.xlabel(f"{class_names[predicted_label]} ({class_names[true_label]})",
              color=color)

def plot_value_array(i, predictions_array, true_labels):
    true_label = true_labels[i]
    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    thisplot = plt.bar(range(10), predictions_array[i], color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array[i])
    
    thisplot[predicted_label].set_color('red')
    thisplot[true_label].set_color('blue')

# Make predictions
predictions = final_model.predict(fashion_x_test)

# Plot some examples
num_rows = 3
num_cols = 5
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictions, fashion_y_test, fashion_x_test)
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictions, fashion_y_test)
plt.tight_layout()
plt.show()

## Conclusion: Hyperparameters Best Practices

After exploring various hyperparameters and their impacts on neural network performance, here are key takeaways and best practices:

### General Hyperparameter Tuning Guidelines

1. **Start Simple**:
   - Begin with default hyperparameters and a simple model
   - Establish a baseline before optimizing

2. **Prioritize Important Hyperparameters**:
   - Learning rate is often the most critical hyperparameter
   - Architecture choices (network depth and width) come next
   - Regularization parameters should be tuned after basic structure is established

3. **Use Systematic Approaches**:
   - Random search often outperforms grid search for the same computational budget
   - Bayesian optimization works well when evaluation is expensive
   - Consider progressive refinement: start broad, then zoom in on promising regions

4. **Validation Strategy Matters**:
   - Always use separate validation data for hyperparameter tuning
   - Consider k-fold cross-validation for smaller datasets
   - Keep a completely separate test set untouched until final evaluation

### Quick Reference: Hyperparameter Ranges

| Hyperparameter | Typical Starting Point | Common Range | Notes |
|----------------|------------------------|--------------|-------|
| Learning Rate | 0.001 | 10⁻⁵ to 10⁻¹ | Log scale search works best |
| Batch Size | 32 or 64 | 16 to 512 | Hardware dependent |
| Hidden Units | 32-128 | Depends on problem | Scale with data complexity |
| Layers | 2-3 | 1 to 10+ | Depends on problem complexity |
| Dropout Rate | 0.2 | 0 to 0.5 | Higher for larger models |
| L2 Regularization | 0.0001 | 10⁻⁵ to 10⁻² | Log scale search |
| Optimizer | Adam | Adam, RMSprop, SGD | Adam works well for most cases |

### Practical Tips

- **Track Your Experiments**: Always use experiment tracking to record hyperparameter settings and results
- **Hardware Considerations**: Some hyperparameters (like batch size) depend on available GPU memory
- **Time vs. Performance Tradeoff**: Consider the computational cost of hyperparameter tuning against potential gains
- **Domain-Specific Knowledge**: Use domain expertise to guide hyperparameter selection when available
- **Ensemble Models**: Consider creating ensembles of models with different hyperparameter settings

Remember that hyperparameter tuning is as much an art as it is a science. What works for one problem or dataset might not work for another. Building intuition through experimentation is key to becoming proficient at deep learning.